# Phase 12 — Post-hoc decision thresholds and INT8 calibration variants (merged configuration)
No retraining: the final SkyShield-IDS (teacher) and SkyShield-IDS-S (student with distillation) are reused.
* Decision thresholds: a per-class offset b_c is added to the log-probabilities, and the predicted class is
  argmax_c (log p_c + b_c). The offsets are tuned on the complete VALIDATION split by coordinate search to
  maximise validation Macro F1; the test split is used only once, with the tuned offsets.
* INT8 calibration variants (TensorFlow Lite full-integer post-training quantization), each evaluated on the test split:
  A. 500 training records stratified by class (as in Phase 8)
  B. 2,000 training records stratified by class and dataset of origin
  C. as B, with test inputs clipped to the training range [0, 1] before quantization
  D. as B, with float32 model input and output (weights and activations remain INT8)
Outputs: outputs_v3/posthoc_v1/ (results/, models/, posthoc_v1_log.txt)

In [ ]:
# %% Cell 0 — mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %% Cell 1 — settings and helpers
import os, json, time, gc, platform
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split

FEAT = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/'
MODELS = {'teacher': '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/models/final_config1_merged.keras',
          'student_kd': '/content/drive/MyDrive/Thesis/outputs_v3/kd_v1/models/kd_config1_merged.keras'}
OUT = '/content/drive/MyDrive/Thesis/outputs_v3/posthoc_v1/'
for sub in ('results/', 'models/'):
    os.makedirs(OUT + sub, exist_ok=True)
CFG, SEED, BATCH = 'config1_merged', 42, 8192
CLASS_NAMES = ['BENIGN', 'Botnet', 'BruteForce', 'DDoS', 'DoS', 'PortScan', 'Spoofing', 'WebAttack']
LOGFILE = OUT + 'posthoc_v1_log.txt'
try:
    Interpreter = tf.lite.Interpreter
except AttributeError:
    os.system('pip -q install ai-edge-litert'); from ai_edge_litert.interpreter import Interpreter

def log(msg):
    line = time.strftime('%Y-%m-%d %H:%M:%S') + ' ' + msg
    print(line, flush=True)
    with open(LOGFILE, 'a') as f:
        f.write(line + '\n')

def metrics(y, pred):
    labels = np.unique(y)
    pr, rc, f1c, sup = precision_recall_fscore_support(y, pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y, pred, labels=labels); tot = cm.sum(); fpr = {}
    for i, k in enumerate(labels):
        fp = cm[:, i].sum() - cm[i, i]; tn = tot - cm[i].sum() - fp; fpr[CLASS_NAMES[k]] = float(fp / (fp + tn))
    b = list(labels).index(0)
    return dict(macro_f1=float(f1c.mean()), accuracy=float(accuracy_score(y, pred)), false_alarm_rate=float(1 - cm[b, b] / cm[b].sum()),
                per_class={CLASS_NAMES[k]: dict(precision=float(p_), recall=float(r_), f1=float(f_), fpr=fpr[CLASS_NAMES[k]], support=int(s_))
                           for k, p_, r_, f_, s_ in zip(labels, pr, rc, f1c, sup)}, confusion_matrix=cm.tolist())

def short(m):
    return (f"Macro F1 {m['macro_f1']:.4f} | accuracy {m['accuracy']:.4f} | false alarm {m['false_alarm_rate'] * 100:.2f}% | per-class F1 "
            + str({k: round(v['f1'], 3) for k, v in m['per_class'].items()}))

def macro_f1_fast(y, pred, k=8):
    cm = np.bincount(y * k + pred, minlength=k * k).reshape(k, k)
    tp = np.diag(cm); p = cm.sum(0); t = cm.sum(1)
    f1 = np.where(p + t > 0, 2 * tp / np.maximum(p + t, 1), 0)
    present = t > 0
    return f1[present].mean()

def tune_offsets(logp, y, grid=np.arange(-4.0, 4.01, 0.25), sweeps=3):
    """Coordinate search for per-class log-probability offsets that maximise Macro F1 (validation split only)."""
    b = np.zeros(logp.shape[1]); best = macro_f1_fast(y, logp.argmax(1))
    for _ in range(sweeps):
        for c in range(logp.shape[1]):
            for g in grid:
                bb = b.copy(); bb[c] = g
                s = macro_f1_fast(y, (logp + bb).argmax(1))
                if s > best + 1e-6: best, b = s, bb
    return b, float(best)

def tflite_predict(path, X, clip=False):
    it = Interpreter(model_path=path); inp, out = it.get_input_details()[0], it.get_output_details()[0]
    scale, zp = inp['quantization']; preds = []
    for s in range(0, len(X), BATCH):
        xb = X[s:s + BATCH]
        if clip: xb = np.clip(xb, 0.0, 1.0)
        if inp['dtype'] == np.int8:
            xb = np.clip(np.round(xb / scale + zp), -128, 127).astype(np.int8)
        it.resize_tensor_input(inp['index'], xb.shape); it.allocate_tensors()
        it.set_tensor(inp['index'], xb); it.invoke()
        preds.append(it.get_tensor(out['index']).argmax(1))
    return np.concatenate(preds)

def convert_int8(model, rep, int_io=True):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([rep[i:i + 1]] for i in range(len(rep)))
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    if int_io:
        conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    return conv.convert()

log('=' * 100)
log(f'Post-hoc v1 | TensorFlow {tf.__version__} | Python {platform.python_version()} | CPU cores {os.cpu_count()}')
Xtr = np.load(FEAT + CFG + '/X_train.npy', mmap_mode='r'); ytr = np.load(FEAT + CFG + '/y_train.npy'); s_tr = np.load(FEAT + CFG + '/source_train.npy')
Xv = np.load(FEAT + CFG + '/X_val.npy').astype(np.float32)[..., None]; yv = np.load(FEAT + CFG + '/y_val.npy').astype(np.int64)
Xte = np.load(FEAT + CFG + '/X_test.npy').astype(np.float32)[..., None]; yte = np.load(FEAT + CFG + '/y_test.npy').astype(np.int64)
log(f'validation {len(yv):,} | test {len(yte):,} | share of test records outside [0, 1] in at least one feature: '
    f'{100 * ((Xte < 0) | (Xte > 1)).any(axis=(1, 2)).mean():.2f}%')

In [ ]:
# %% Cell 2 — post-hoc decision thresholds (offsets tuned on the validation split)
for kind, path in MODELS.items():
    rpath = OUT + f'results/thresholds_{kind}.json'
    if os.path.exists(rpath):
        log(f'thresholds {kind}: already done, skipping'); continue
    model = keras.models.load_model(path, compile=False)
    lv = np.log(np.clip(model.predict(Xv, batch_size=BATCH, verbose=0), 1e-9, 1))
    lt = np.log(np.clip(model.predict(Xte, batch_size=BATCH, verbose=0), 1e-9, 1))
    base_val = macro_f1_fast(yv, lv.argmax(1))
    b, tuned_val = tune_offsets(lv, yv)
    m0, m1 = metrics(yte, lt.argmax(1)), metrics(yte, (lt + b).argmax(1))
    log(f'thresholds {kind}: validation Macro F1 {base_val:.4f} -> {tuned_val:.4f} | offsets {dict(zip(CLASS_NAMES, np.round(b, 2)))}')
    log(f'thresholds {kind} TEST default: {short(m0)}')
    log(f'thresholds {kind} TEST tuned:   {short(m1)}')
    json.dump(dict(model=kind, offsets=dict(zip(CLASS_NAMES, b.tolist())), val_macro_f1_default=float(base_val),
                   val_macro_f1_tuned=tuned_val, test_default=m0, test_tuned=m1), open(rpath, 'w'), indent=2)
    del model; gc.collect()

In [ ]:
# %% Cell 3 — INT8 calibration variants
idxA, _ = train_test_split(np.arange(len(ytr)), train_size=500, stratify=ytr, random_state=SEED)
strata = np.char.add(ytr.astype(str), s_tr.astype(str))
idxB, _ = train_test_split(np.arange(len(ytr)), train_size=2000, stratify=strata, random_state=SEED)
REP = {'A': np.asarray(Xtr[np.sort(idxA)], np.float32)[..., None], 'B': np.asarray(Xtr[np.sort(idxB)], np.float32)[..., None]}
VARIANTS = [('A', 'A', True, False), ('B', 'B', True, False), ('C', 'B', True, True), ('D', 'B', False, False)]   # (name, rep set, int8 I/O, clip)
for kind, path in MODELS.items():
    model = keras.models.load_model(path, compile=False)
    p_f32 = model.predict(Xte, batch_size=BATCH, verbose=0).argmax(1)
    for vname, rep, int_io, clip in VARIANTS:
        rpath = OUT + f'results/int8_{kind}_{vname}.json'
        if os.path.exists(rpath):
            log(f'INT8 {kind} {vname}: already done, skipping'); continue
        tfl = convert_int8(model, REP[rep], int_io); mpath = OUT + f'models/{kind}_int8_{vname}.tflite'; open(mpath, 'wb').write(tfl)
        p = tflite_predict(mpath, Xte, clip=clip); m = metrics(yte, p)
        m.update(variant=vname, representative=rep, representative_records=int(len(REP[rep])), int8_io=int_io, clip_inputs=clip,
                 size_kb=round(len(tfl) / 1024, 1), agreement_with_float32=float((p == p_f32).mean()))
        json.dump(m, open(rpath, 'w'), indent=2)
        log(f'INT8 {kind} {vname} ({len(REP[rep])} records, int8 I/O {int_io}, clip {clip}): {short(m)} | '
            f'agreement {100 * m["agreement_with_float32"]:.1f}% | {m["size_kb"]} KB')
    del model; gc.collect()
log('post-hoc v1: done')

In [ ]:
# %% Cell 4 — summary
rows = []
for kind in MODELS:
    p = OUT + f'results/thresholds_{kind}.json'
    if os.path.exists(p):
        r = json.load(open(p))
        rows.append(dict(model=kind, setting='float32 default', macro_f1=r['test_default']['macro_f1'], false_alarm=r['test_default']['false_alarm_rate']))
        rows.append(dict(model=kind, setting='float32 tuned offsets', macro_f1=r['test_tuned']['macro_f1'], false_alarm=r['test_tuned']['false_alarm_rate']))
    for v, *_ in VARIANTS:
        p = OUT + f'results/int8_{kind}_{v}.json'
        if os.path.exists(p):
            r = json.load(open(p)); rows.append(dict(model=kind, setting=f'INT8 {v}', macro_f1=r['macro_f1'], false_alarm=r['false_alarm_rate'],
                                                     agreement=r['agreement_with_float32'], size_kb=r['size_kb']))
print(pd.DataFrame(rows).round(4).to_string(index=False))